In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.core import image_stats
from dataeval.data import Limit, View
from dataeval.flags import ImageStats

# Statistics are calculated across a process pool. Capping it keeps this example's memory
# footprint predictable; leave it unset to use every available core.
set_max_processes(4)

In [3]:
# Load the SeaDrone dataset
sd_dataset = SeaDrone(root="./data", image_set="val", download=True)

# Limit to first 50 images for demonstration
dataset = View(sd_dataset, Limit(50))

In [4]:
results_background = image_stats(
    data=dataset,
    stats=ImageStats.VISUAL_SHARPNESS,
    per_image=True,
    per_target=True,
    per_background=True,
    normalize_pixel_values=False,
)

print(f"Computed statistics: {sorted(results_background['stats'])}")

Computed statistics: ['background_fraction', 'background_sharpness', 'sharpness']


In [5]:
# The result spans image rows and target rows; only the image rows carry a background, so the
# target rows hold NaN here and have to be dropped before the fraction can be summarized.
fraction = pl.Series(results_background["stats"]["background_fraction"]).drop_nulls().drop_nans()

print(f"background_fraction: n={len(fraction)}  min={fraction.min():.3f}  median={fraction.median():.3f}")

background_fraction: n=50  min=0.968  median=0.998


In [6]:
metadata = Metadata(dataset)
supplied = set(metadata.factor_names)
metadata.add_factors(results_background)

print("levels      :", metadata.levels)
print("level counts:", metadata.level_counts)
print("new factors :", sorted(set(metadata.factor_names) - supplied))

levels      : ('unit', 'instance')
level counts: {'unit': 50, 'instance': 143}
new factors : ['instance_sharpness', 'unit_background_fraction', 'unit_background_sharpness', 'unit_sharpness']


/tmp/ipykernel_4893/2838730580.py:2: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  supplied = set(metadata.factor_names)


In [7]:
print("dropped:", metadata.dropped_factors)

print(
    metadata
    .rows_at("unit")
    .select("item_index", "unit_sharpness", "unit_background_sharpness", "unit_background_fraction")
    .head(5)
)

dropped: {'latitude': ['mixed_types'], 'longitude': ['mixed_types'], 'date_time': ['cardinality_over_budget'], 'instance_background_sharpness': ['no_values_at_level'], 'instance_background_fraction': ['no_values_at_level']}
shape: (5, 4)
┌────────────┬────────────────┬───────────────────────────┬──────────────────────────┐
│ item_index ┆ unit_sharpness ┆ unit_background_sharpness ┆ unit_background_fraction │
│ ---        ┆ ---            ┆ ---                       ┆ ---                      │
│ i64        ┆ f32            ┆ f32                       ┆ f32                      │
╞════════════╪════════════════╪═══════════════════════════╪══════════════════════════╡
│ 0          ┆ 25.101564      ┆ 25.021826                 ┆ 0.999166                 │
│ 1          ┆ 21.477745      ┆ 21.207449                 ┆ 0.997232                 │
│ 2          ┆ 19.58268       ┆ 19.114891                 ┆ 0.997434                 │
│ 3          ┆ 19.48291       ┆ 19.3801                   ┆ 0.9993

/tmp/ipykernel_4893/2922025144.py:5: UserWarning: `altitude`, `frame`, `instance_sharpness`, `object_id`, `object_size`, `unit_background_fraction`, `unit_background_sharpness` and `unit_sharpness` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  .rows_at("unit")


In [8]:
units = metadata.rows_at("unit")
sharper = units.select((pl.col("unit_sharpness") > pl.col("unit_background_sharpness")).sum()).item()

print(f"images sharper as a whole than their own background: {sharper}/{len(units)}")
print(f"mean sharpness  object={metadata.rows_at('instance')['instance_sharpness'].mean():.1f}")
print(f"                image ={units['unit_sharpness'].mean():.1f}")
print(f"                back  ={units['unit_background_sharpness'].mean():.1f}")

images sharper as a whole than their own background: 48/50
mean sharpness  object=55.3
                image =22.5
                back  =22.0


In [9]:
# TEST ASSERTION CELL ###
# Verify the statistics and the regions they were measured over
import numpy as np

assert set(results_background["stats"]) == {"sharpness", "background_sharpness", "background_fraction"}
assert fraction.to_numpy().min() > 0.9  # nearly all of every image survives masking on SeaDrone
assert np.median(fraction.to_numpy()) > 0.99

# Verify where the values landed: three sharpness factors and a background fraction, and no background on a detection
assert metadata.levels == ("unit", "instance")
assert metadata.level_counts["unit"] == len(dataset) == 50
assert metadata.level_counts["instance"] > 50
assert set(metadata.factor_names) - supplied == {
    "instance_sharpness",
    "unit_sharpness",
    "unit_background_sharpness",
    "unit_background_fraction",
}
assert "instance_background_sharpness" in metadata.dropped_factors